Import libraries

In [1]:
import os
import sys
import copy
import time
import importlib
import numpy as np
import pandas as pd
import tensorflow as tf
import joblib as jl
threads = 4

from tqdm.notebook import tqdm

In [2]:
import matplotlib as mpl
from matplotlib import pyplot as plt
from matplotlib import ticker
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
%matplotlib qt

# # Use custom style
# # See: https://matplotlib.org/1.5.3/users/style_sheets.html
# style_name = 'paper_1column'
# mpl.style.use(
#     os.environ['WORKSPACE_PATH'] \
#         + '/RESEARCH/styles/matplotlib/' \
#         + style_name + '.mplstyle'
# )
# # Get colors list
# COLORS = mpl.rcParams['axes.prop_cycle'].by_key()['color']*10

In [3]:
T = 10000

cwd = os.getcwd()
figs = cwd + f'/figs_T{T}K'
try:
    os.mkdir(figs)
except:
    pass

Define paths

In [4]:
sts_fld = cwd + f'/../../isothermal/O3/nestedTimescales/postprocess/CG-DeepONet/src/sts_cg_park/T{T}K'
sts_file = sts_fld + '/sts/excit_diss/output_box/box.dat'
pop_file = sts_fld + '/sts/excit_diss/output_box/pop_O2.dat'
lev_file = '/home/zanardi/WORKSPACE/database/levels/O2.csv'

Manipulate energy levels

In [5]:
lev_dtb = pd.read_csv(lev_file)
lev_dtb['EVib'] = lev_dtb['EVib'] - np.amin(lev_dtb['EVib'])
lev_dtb['ERot'] = lev_dtb['ERot'] - np.amin(lev_dtb['ERot'])
lev_dtb['E'] = lev_dtb['EVib'] + lev_dtb['ERot']
lev_dtb['g'] = 3./2.*(2.*lev_dtb['jqn'] + 1.)
n_lev = len(lev_dtb['g'])

Get StS solution

In [6]:
# States number densities
def n_states(instants, file):
    with open(file, "r") as f:
        lines = f.readlines()
        del lines[:1]
    states = []
    for i in tqdm(range(instants)):
        states.append(np.loadtxt(lines[i*(n_lev+1)+1:(n_lev+1)*(i+1)], dtype=float).T[1].reshape(1,-1))
    return np.vstack(states)

# Solution from StS
data = pd.read_csv(sts_file, sep="  ", header=1, engine='python')
t = data[[col for col in data.columns if 't' in col]].to_numpy()
n_sts = n_states(instants=len(data), file=pop_file)
n_sts_ln = np.log10(n_sts)

  0%|          | 0/1421 [00:00<?, ?it/s]

3D plot

In [7]:
style = dict(
    markeredgewidth=1,
    lw=1.,
    linestyle='',
    markersize=1,
    marker='x',
    c='k'
)

In [12]:
fig = plt.figure()
ax = plt.axes(projection='3d')
# Setting up axis
# X axis
ax.set_xlabel(r'$E_{vib}$ [eV]')
# Y axis
ax.set_ylabel(r'$E_{rot}$ [eV]')
# Z axis
ax.set_zlabel(r'$\log_{10}(n_i/g_i)$')
# Plot
ax.plot3D(lev_dtb['EVib'], lev_dtb['ERot'], n_sts_ln[200], **style)
plt.grid()
plt.show()

3D animation

In [9]:
# Initialize lines for levels distribution
def init_lines(
    ax,
    axim=0.
):
    # Set up axes
    ax.set_xlabel(r'$E_{vib}$ [eV]')
    ax.set_ylabel(r'$E_{rot}$ [eV]')
    ax.set_zlabel(r'$\log_10(n_i/g_i)$')
    # Initialize lines.Line2D objects
    lines = [ax.plot3D([],[],[],**style)[0]]
    ax.view_init(elev=30, azim=axim)
    return ax, lines

# Create animation
def get_animation(
    t_vec,
    n_sts,
    frames,
    axim=0.
):
    # Initialize a figure in which the graphs will be plotted
    fig = plt.figure()
    ax = fig.add_subplot(projection="3d")
    # Initialize levels distribution lines objects
    ax, lines = init_lines(ax, axim=axim)
    xlim = [np.amin(lev_dtb['EVib']), np.amax(lev_dtb['EVib'])]
    ylim = [np.amin(lev_dtb['ERot']), np.amax(lev_dtb['ERot'])]
    
    def animate(frame):
        idx = int(frame/frames*len(t_vec))
        # Plot distribution
        lines[0].set_xdata(lev_dtb['EVib'])
        lines[0].set_ydata(lev_dtb['ERot'])
        lines[0].set_3d_properties(n_sts_ln[idx])
        # Rescale axis limits
        ax.set_xlim(xlim)
        ax.set_ylim(ylim)
        zlim = [np.amin(n_sts_ln[idx]), np.amax(n_sts_ln[idx])]
        ax.set_zlim(zlim)
        return lines
        
    # Get animation
    return FuncAnimation(
        fig,
        animate,
        frames=frames,
        blit=True
    )


In [10]:
frames = 100

for i, axim in enumerate([-60,0,60]):
    anim = get_animation(
        t_vec=t,
        n_sts=n_sts,
        frames=frames,
        axim=axim
    )

    # Save animation
    anim.save(f'{figs}/view_{i}.mp4', fps=frames/10, dpi=600)

    # Display animation
    HTML(anim.to_jshtml())